In [48]:
import pandas as pd
import json
import jiwer
import ast
from jiwer import cer, wer
from difflib import SequenceMatcher

#### Ground Truth

In [49]:
# Load data frame
df_ground_truth = pd.read_json(r"C:\Users\Bvarta-Aditya\Workspace\fnb_menu\data_raw\benchmark_processed_images/ground_truth.json")
df_ground_truth.head()

,poi_id,image,total_items,items
0,poiabc123,poiabc123_foto_14_.jpg,1,"[{'name': 'Maze Ramen', 'price': 21000}]"
1,poiabc123,poiabc123_foto_19_Ilham_Pratama.jpg,15,"[{'name': 'Lychee Tea', 'price': 18000}, {'nam..."
2,poiabc123,poiabc123_foto_28_Heppi_Elastiko.jpg,24,"[{'name': 'Tori Paitan Ramen', 'prices': [{'am..."
3,poiabc123,poiabc123_foto_49_Gitta_Fatima_Rahayu.jpg,2,"[{'name': 'Spicy chicken roll', 'price': 35000..."
4,poiabc123,poiabc123_foto_57_Ilham_Pratama.jpg,2,"[{'name': 'Beef Curry Rice', 'prices': [{'amou..."


#### OCR results

In [50]:
# Load data frame
df_ocr = pd.read_json(r"C:\Users\Bvarta-Aditya\Workspace\fnb_menu\data_raw\benchmark_processed_images/ocr_results.json")
df_ocr.head()

,poi_id,image,total_items,items
0,poiabc123,poiabc123_foto_14_.jpg,1,"[{'name': '6', 'price': 21000}]"
1,poiabc123,poiabc123_foto_19_Ilham_Pratama.jpg,15,"[{'name': 'LycheeTea', 'price': 18000}, {'name..."
2,poiabc123,poiabc123_foto_28_Heppi_Elastiko.jpg,24,"[{'name': 'Tori Paitan Ramen', 'prices': [{'am..."
3,poiabc123,poiabc123_foto_49_Gitta_Fatima_Rahayu.jpg,2,"[{'name': 'Spicy chicken roll', 'price': 35000..."
4,poiabc123,poiabc123_foto_57_Ilham_Pratama.jpg,2,"[{'name': 'Beef Curry Rice', 'prices': [{'amou..."


#### OCR Results Evaluation Per Image

Calculate similarity

In [51]:
# Similarity function
def calculate_similarity(text1: str, text2: str) -> float:
    """Calculate similarity between two strings (0-1)."""
    return SequenceMatcher(None, text1.lower(), text2.lower()).ratio()

JSON accuracy score

In [52]:
# Accuracy function
def json_accuracy(ground_truth: list, predicted: list) -> float:
    """
    OmniAI-style JSON accuracy.
    Compare field by field between ground truth and predicted items.
    """
    def flatten_item(item: dict, prefix='') -> dict:
        """Flatten nested dict/list into dot-notation keys."""
        result = {}
        for k, v in item.items():
            key = f"{prefix}.{k}" if prefix else k
            if isinstance(v, list):
                for i, val in enumerate(v):
                    if isinstance(val, dict):
                        result.update(flatten_item(val, f"{key}[{i}]"))
                    else:
                        result[f"{key}[{i}]"] = val
            elif isinstance(v, dict):
                result.update(flatten_item(v, key))
            else:
                result[key] = v
        return result

    total   = 0
    correct = 0

    # Match items by name similarity
    used = set()
    for gt_item in ground_truth:
        best_match  = None
        best_sim    = 0

        for i, pred_item in enumerate(predicted):
            if i in used:
                continue
            sim = calculate_similarity(
                gt_item.get('name', ''),
                pred_item.get('name', '')
            )
            if sim > best_sim:
                best_sim   = sim
                best_match = (i, pred_item)

        if best_match and best_sim >= 0.7:
            used.add(best_match[0])
            gt_flat   = flatten_item(gt_item)
            pred_flat = flatten_item(best_match[1])

            for key, val in gt_flat.items():
                total += 1
                if pred_flat.get(key) == val:
                    correct += 1
        else:
            # No match found — count all fields as wrong
            total += len(flatten_item(gt_item))

    accuracy = round(correct / total * 100, 2) if total else 0.0
    return accuracy

- CER: Character Error Rate
- WER: Word Error Rate

In [53]:
# Metrics per image function
def calculate_metrics_per_image(df_ground_truth, df_ocr) -> pd.DataFrame:
    results = []

    for _, gt_row in df_ground_truth.iterrows():
        image   = gt_row['image']
        ocr_row = df_ocr[df_ocr['image'] == image]
        if ocr_row.empty:
            continue
        ocr_row = ocr_row.iloc[0]

        gt_items   = gt_row['items']
        pred_items = ocr_row['items']

        gt_names   = ' '.join([i['name'] for i in gt_items])
        pred_names = ' '.join([i['name'] for i in pred_items])
        gt_prices  = ' '.join([str(i.get('price', '')) for i in gt_items])
        pred_prices = ' '.join([str(i.get('price', '')) for i in pred_items])

        results.append({
            'image':        image,
            'gt_total':     gt_row['total_items'],
            'pred_total':   ocr_row['total_items'],
            'item_diff':    ocr_row['total_items'] - gt_row['total_items'],
            'name_CER':     round(cer(gt_names,  pred_names)  * 100, 2),
            'name_WER':     round(wer(gt_names,  pred_names)  * 100, 2),
            'price_CER':    round(cer(gt_prices, pred_prices) * 100, 2),
            'price_WER':    round(wer(gt_prices, pred_prices) * 100, 2),
            'json_accuracy': json_accuracy(gt_items, pred_items),  # ← OmniAI style
        })

    df_metrics = pd.DataFrame(results)

    print(f"Avg name CER:      {df_metrics['name_CER'].mean():.2f}%")
    print(f"Avg name WER:      {df_metrics['name_WER'].mean():.2f}%")
    print(f"Avg price CER:     {df_metrics['price_CER'].mean():.2f}%")
    print(f"Avg price WER:     {df_metrics['price_WER'].mean():.2f}%")
    print(f"Avg JSON accuracy: {df_metrics['json_accuracy'].mean():.2f}%")

    return df_metrics

In [54]:
# Use the function
df_metrics = calculate_metrics_per_image(df_ground_truth, df_ocr)
df_metrics

Avg name CER:      23.46%
Avg name WER:      30.20%
Avg price CER:     1.31%
Avg price WER:     5.32%
Avg JSON accuracy: 71.86%


,image,gt_total,pred_total,item_diff,name_CER,name_WER,price_CER,price_WER,json_accuracy
0,poiabc123_foto_14_.jpg,1,1,0,100.00,100.00,0.00,0.00,0.00
1,poiabc123_foto_19_Ilham_Pratama.jpg,15,15,0,22.96,37.50,6.98,26.67,86.67
2,poiabc123_foto_28_Heppi_Elastiko.jpg,24,24,0,4.48,10.34,0.90,5.26,94.52
3,poiabc123_foto_49_Gitta_Fatima_Rahayu.jpg,2,2,0,0.00,0.00,0.00,0.00,100.00
4,poiabc123_foto_57_Ilham_Pratama.jpg,2,2,0,13.33,33.33,0.00,0.00,50.00
5,poiabc123_foto_63_Wahyu_Ramadhan.jpg,2,2,0,0.00,0.00,0.00,0.00,100.00


## Load Final Output

#### Combined Ground Truth

In [76]:
# Load data frame
df_full_ground_truth = pd.read_json(r"C:\Users\Bvarta-Aditya\Workspace\fnb_menu\data_raw\benchmark_processed_images/ground_truth_combined.json")
df_full_ground_truth.head()

,poi_id,total_items,items
0,poiabc123,44,"{'name': 'Maze Ramen', 'price': 21000}"
1,poiabc123,44,"{'name': 'Lychee Tea', 'price': 18000}"
2,poiabc123,44,"{'name': 'Ocha Ice / Hot', 'price': 8000}"
3,poiabc123,44,"{'name': 'Lemon Tea', 'price': 18000}"
4,poiabc123,44,"{'name': 'Teran Orenji', 'price': 19000}"


#### Combined OCR Results
- After cleaning LLM and deduplication

In [77]:
# Load data frame
df_full_ocr = pd.read_json(r"C:\Users\Bvarta-Aditya\Workspace\fnb_menu\data_raw\benchmark_processed_images/ocr_menu_results_20260506_123612.json")
df_full_ocr.head()

,poi_id,total_items,items
0,poiabc123,43,"{'name': 'Air Mineral', 'price': 7000}"
1,poiabc123,43,"{'name': 'Ajitsuke Tamago', 'price': 5000}"
2,poiabc123,43,"{'name': 'Avocado Coffee', 'price': 30000}"
3,poiabc123,43,"{'name': 'Avocado Juice', 'price': 16000}"
4,poiabc123,43,"{'name': 'Beef Curry Rice', 'prices': [{'amoun..."


#### OCR Results Evaluation Per POI

In [87]:
# Evaluate per poi function
def evaluate_per_poi(df_ground_truth, df_ocr) -> pd.DataFrame:
    results = []
    poi_ids = df_ground_truth['poi_id'].unique()

    for poi_id in poi_ids:
        gt_rows  = df_ground_truth[df_ground_truth['poi_id'] == poi_id]
        ocr_rows = df_ocr[df_ocr['poi_id'] == poi_id]

        if ocr_rows.empty:
            print(f"No OCR results for poi_id={poi_id}")
            continue

        # Sort by name before comparing ← CER/WER position independent
        gt_names_sorted   = sorted(gt_rows['name'].fillna('').tolist())
        pred_names_sorted = sorted(ocr_rows['name'].fillna('').tolist())

        gt_prices_sorted   = sorted(gt_rows['price'].fillna('').astype(str).tolist())
        pred_prices_sorted = sorted(ocr_rows['price'].fillna('').astype(str).tolist())

        gt_names    = ' '.join(gt_names_sorted)
        pred_names  = ' '.join(pred_names_sorted)
        gt_prices   = ' '.join(gt_prices_sorted)
        pred_prices = ' '.join(pred_prices_sorted)

        # For json_accuracy and recall — sort items by name
        gt_items   = sorted(gt_rows[['name', 'price']].to_dict(orient='records'),
                           key=lambda x: x.get('name', ''))
        pred_items = sorted(ocr_rows[['name', 'price']].to_dict(orient='records'),
                           key=lambda x: x.get('name', ''))

        results.append({
            'poi_id':        poi_id,
            'gt_total':      len(gt_rows),
            'pred_total':    len(ocr_rows),
            'item_diff':     len(ocr_rows) - len(gt_rows),
            'name_CER':      round(cer(gt_names,   pred_names)   * 100, 2),
            'name_WER':      round(wer(gt_names,   pred_names)   * 100, 2),
            'price_CER':     round(cer(gt_prices,  pred_prices)  * 100, 2),
            'price_WER':     round(wer(gt_prices,  pred_prices)  * 100, 2),
            'json_accuracy': json_accuracy(gt_items, pred_items),
        })

    df_metrics = pd.DataFrame(results)

    print(f"Avg name CER:      {df_metrics['name_CER'].mean():.2f}%")
    print(f"Avg name WER:      {df_metrics['name_WER'].mean():.2f}%")
    print(f"Avg price CER:     {df_metrics['price_CER'].mean():.2f}%")
    print(f"Avg price WER:     {df_metrics['price_WER'].mean():.2f}%")
    print(f"Avg JSON accuracy: {df_metrics['json_accuracy'].mean():.2f}%")

    return df_metrics

In [86]:
# Extract name and price from items dict column
df_full_ocr['name']  = df_full_ocr['items'].apply(lambda x: x.get('name', ''))
df_full_ocr['price'] = df_full_ocr['items'].apply(lambda x: x.get('price', ''))

df_full_ground_truth['name']  = df_full_ground_truth['items'].apply(lambda x: x.get('name', ''))
df_full_ground_truth['price'] = df_full_ground_truth['items'].apply(lambda x: x.get('price', ''))

# Evaluate the metrics
df_metrics = evaluate_per_poi(df_full_ground_truth, df_full_ocr)
df_metrics

Avg name CER:      7.03%
Avg name WER:      9.71%
Avg price CER:     3.70%
Avg price WER:     8.11%
Avg JSON accuracy: 87.50%


,poi_id,gt_total,pred_total,item_diff,name_CER,name_WER,price_CER,price_WER,json_accuracy
0,poiabc123,44,43,-1,7.03,9.71,3.7,8.11,87.5
